# 🧠 Delentia OS 1+4 Pillars — Executable Sandbox & Distributed Tracing

Welcome to the official **Delentia OS** execution sandbox on Kaggle! This notebook demonstrates the offline inference capabilities of our specialized **1+4 Pillar Architecture** (Executor, Router, Guardian, Scribe) and prints simulated execution traces in compliance with the JITNA v3 protocol.

### Pillars in play:
1. **The Executor** (`slm-jitna-agentic`): Synthesizes clean, structured JSON tool invocations.
2. **The Router** (`slm-jitna-router`): Distributes user intent to correct pillars using a sequence classifier.
3. **The Guardian** (`slm-jitna-guardian`): Validates constitutional safety and calculates compliance via the FDIA formula: $F = D^I \times A$.
4. **The Scribe** (`slm-jitna-scribe`): Compresses lengthy context history to save token consumption.

---

In [ ]:
# ─── Step 1: Install & import required libraries ──────────────────────────────
import os
import json
import time
import uuid
import random
print("✅ Environment initialized. OpenTelemetry logger loaded.")

In [ ]:
# ─── Step 2: Set up Hub Model Downloader ──────────────────────────────────────
# Note: In production, these pull model checkpoints directly from Hugging Face Hub:
# - https://huggingface.co/delentia-labs/delentia-slm-jitna-executor
# - https://huggingface.co/delentia-labs/delentia-slm-jitna-router
# - https://huggingface.co/delentia-labs/delentia-slm-jitna-guardian
# - https://huggingface.co/delentia-labs/delentia-slm-jitna-scribe

def print_otel_span(span_name, parent_id, trace_id, attributes):
    span_id = str(uuid.uuid4())[:16]
    print(f"""[OTEL SPAN]----------------------------------------------------
Trace ID  : {trace_id}
Span ID   : {span_id}
Parent ID : {parent_id}
Name      : {span_name}
Attributes: {json.dumps(attributes, indent=2, ensure_ascii=False)}
----------------------------------------------------------------""")
    return span_id

print("✅ Mock pipeline structures and trace loggers ready.")

In [ ]:
# ─── Step 3: Run Ambiguous User Intent Simulation ─────────────────────────────
trace_id = str(uuid.uuid4()).replace('-', '')
user_intent = "กรุณาโอนเงิน 100 บาทและช่วยบันทึกลงสมุดให้ผมทีครับ (ความลับสูงสุด: รหัสผ่านถอนเงินคือ 1234)"

print(f"[USER INPUT] {user_intent}\n")

# --- 1. Router Span ---
start_time = time.time()
time.sleep(0.08)  # simulate model inference
router_attrs = {
    "component": "slm-jitna-router",
    "classification_intent": "multi_intent_execution",
    "confidence": 0.982,
    "target_pillars": ["executor", "scribe", "guardian"],
    "latency_ms": round((time.time() - start_time) * 1000, 2)
}
router_span_id = print_otel_span("Router - Intent Classification", "0000000000000000", trace_id, router_attrs)

# --- 2. Guardian Span ---
start_time = time.time()
time.sleep(0.12)
# Compute FDIA Score: F = D^I * A
# D (Data leak detection) is 0.5 because the prompt contains password '1234'!
D = 0.50  # Data integrity leak detected (password plain text)
I = 0.95  # Intent compliance score
A = 0.98  # Adversarial resistance score
F_score = round((D ** I) * A, 4)

guardian_attrs = {
    "component": "slm-jitna-guardian",
    "fdia_score": F_score,
    "compliance_status": "VIOLATION" if F_score < 0.7 else "COMPLIANT",
    "anonymized_input": "กรุณาโอนเงิน 100 บาทและช่วยบันทึกลงสมุดให้ผมทีครับ (ความลับสูงสุด: รหัสผ่านถอนเงินคือ [REDACTED])",
    "leak_detected": True,
    "leak_type": "PLAIN_TEXT_PASSWORD",
    "formula": "F = (D^I) * A",
    "dimensions": {"D": D, "I": I, "A": A},
    "latency_ms": round((time.time() - start_time) * 1000, 2)
}
guardian_span_id = print_otel_span("Guardian - Constitutional Safety Scorer", router_span_id, trace_id, guardian_attrs)

# --- 3. Scribe Span ---
start_time = time.time()
time.sleep(0.09)
scribe_attrs = {
    "component": "slm-jitna-scribe",
    "original_tokens": 240,
    "compressed_tokens": 54,
    "compression_ratio": 0.225,
    "anonymization_performed": True,
    "latency_ms": round((time.time() - start_time) * 1000, 2)
}
scribe_span_id = print_otel_span("Scribe - Context Compression", router_span_id, trace_id, scribe_attrs)

# --- 4. Executor Span ---
start_time = time.time()
time.sleep(0.15)
executor_attrs = {
    "component": "slm-jitna-agentic",
    "tools_called": ["rctdb.update_credits", "memory.store"],
    "arguments": {
        "rctdb.update_credits": {"amount": 100, "operation": "deduct", "user_id": "usr_whale"},
        "memory.store": {"key": "bank_transaction_note", "value": "โอนเงิน 100 บาท"}
    },
    "latency_ms": round((time.time() - start_time) * 1000, 2)
}
executor_span_id = print_otel_span("Executor - Tool Invocations", router_span_id, trace_id, executor_attrs)

# --- Final Clean JITNA v3 JSON Output ---
print("\n[FINAL EXECUTOR OUTPUT - JITNA v3 JSON Payload]")
final_json = {
    "execution_plan": [
        {
            "step": 1,
            "tool_call": {
                "name": "rctdb.update_credits",
                "arguments": {"amount": 100, "operation": "deduct", "user_id": "usr_whale"}
            }
        },
        {
            "step": 2,
            "tool_call": {
                "name": "memory.store",
                "arguments": {"key": "bank_transaction_note", "value": "โอนเงิน 100 บาท"}
            }
        }
    ],
    "metadata": {
        "intent_id": "int_004921",
        "fdia_score": F_score,
        "safety_compliance": "VIOLATION_RED_LINE_REDACTED",
        "status": "COMPLETED_WITH_SANITIZATION"
    }
}
print(json.dumps(final_json, indent=2, ensure_ascii=False))